In [ ]:
!pip install pandas==3.0.2

# Opentron Protocol Workflow - OT-IVGC
## in vitro growth characterization of microbial biofertilizers

- **Document Name:** Opentron Protocol Workflow - OT-IVGC
- **Version:** 1.0
- **Effective date:** 2026-04-27
- **Author:** Rasmus Tøffner-Clausen
- **Opentron Unit operations (ordered):** 
    - `U01-Onboarding`
    - `U02-Washing` 
    - `U03-Normalization`
    - `U04.1-Transfer_lq` & `U04.2-Transfer_s`

# Purpose and scope
- **Purpose:** Assess in vitro growth characteristics of rhizobium based on ARE media comparison.
- **Scope:** Starts from glycerol stock cultures and ends with randomized liquid plates for spectrophotometer incubation and plates for stamping with the Rotor+ and monitoring of solid growth using the Reshape imaging device.

# Steps:
1. **U01-Onboarding**: Culture onboarding from stock
2. **U02-Washing**: Cell washing.
3. **U03-Normalization**: OD Normalization from csv file.
4. **U04.1-Transfer_lq & U04.2-Transfer_s**: Randomized layout media and strain transfer using generated csv file.

## Day plan

***Day 0*** <br>
Run **U01-Onboarding:** ***OT-2 8-Channel mount*** <br>
**Requirements:**
- Glycerol stock
- Desired liquid incubation media (TY)
- Select desired columns from stock columns <br>

**Output**:
Deep well plate with 2 replicates of selected columns. Stock col 1 $\rightarrow$ Dest col 1 & 2. There's therefore a hardlimit on 6 stock columns per deepwell plate in this step.

---

***Day 2*** <br>
Run: **U02-Washing:** ***OT-2 8-Channel mount*** <br>
**Requirements**:
- Incubated deep well plate.
- Select columns: All inculated columns including replicate columns from onboarding

**Output**: Resuspended and pooled deep well plate and 20x dilution within 200 $\mu L$ plate for OD reading. The pooling is done per strain so incubated Dest col 1 & 2 get pooled in dest col 1. This means that source col 2 is pooled in dest col 3 and so forth.

*Note:* Due to difficulties with equipment this protocol has not been ran and instead manual washing has been done. However in relation to another project the central idea has been confirmed to work and is therefore provided. 

---

Run: `od_normalization` cell found in this notebook. <br>
**Requirements**:
- OD data file. (Aquireable from the 20x dilution 200 $\mu L$ plate prepared)

**Output**:
- OD.csv file for use with **U03-Normalization** to obtain desired OD value. For this project it is an OD of 1.
---

Run: **U03-Normalization:** ***OT-2 Single-channel mount*** <br>
**Requirements**
- OD.csv file from `od_normalization`
- Washed, pooled and resuspended deep-well plate.
- Dilution media. ($0.9\%$ NaCl)

**Output**
- Normalized deep well plate containing approx 800 $\mu L$ culture
---

Run: `generate_randomized_mapping` cell found in this notebook <br>
**Requirements**
- Glycerol layout file
- Selected source plate columns
- List of media, (default and maximum: 3).
- Replicate count, (default: 4)

**Output**:
- csv file with strain origin and randomized layout. Another file is generated to serve as a layout reference for data analysis.
---

Run: **U04.1-Transfer_lq:** ***OT-2 Single-channel mount*** <br>
**Requirements**:
- Normalized OD deep-well plate
- Randomized layout in the form of csv file like the one generated with `generate_randomized_mapping`

**Output**:
- Liquid plate for monitoring with spectrophotometer.
---

Run: **U04.2-Transfer_s:** ***OT-2 8-Channel mount*** <br>
**Requirements**:
- Normalized OD deep-well plate
- $0.9%$ NaCl for dilution

**Output**:
- 10x Liquid plate for stamping onto solid agar media using Rotor+ "Spot many" program and growth monitoring using Reshape imaging device.

---

### Imports

Imports required python modules to run the notebook.

In [1]:
import OT_IVGC

# User Configurations

## Parameters
**Requirements:** Set run parameters here.

In [2]:
# --- Layout file path ---
file_path = "examples/glycerol_layout.xlsx"

# --- Columns selected for incubation ---
# --- Valid entries "1"-"6"
cols = ["1","2"]

# --- Media to use ---
# Currently intended for 3 media 4 replicates. This should not be altered but media can be exchanged.
medias = ("CRE","LRE","TY")
replicate_count = 4 # For each media

## Double check source layout

Ensure layout is correctly imported for selected columns. Well is original source well from glycerol stock while `deepwell_well` is the actual well after first running incubation and washing.

In [3]:
filtered_layout = OT_IVGC.get_layout_reference(file_path,
                     selected_cols=cols)

filtered_layout

,IBT #,well,row,col,deepwell_well,deepwell_col
0,IBT 115000,A1,A,1,A1,1
1,IBT 115008,A2,A,2,A3,3
2,IBT 115002,B1,B,1,B1,1
3,IBT 115009,B2,B,2,B3,3
4,IBT 115003,C1,C,1,C1,1
5,IBT 115010,C2,C,2,C3,3
6,IBT 115004,D1,D,1,D1,1
7,IBT 115014,D2,D,2,D3,3
8,IBT 115007,E1,E,1,E1,1
9,IBT 115011,E2,E,2,E3,3


# OD-Normalization

## Parameters for normalization

In [4]:
# --- OD reading File path ---
od_file_path = "examples/od_wash.xlsx" # <- File path to read diluted plate.

# --- OD Parameters ---
od_target = 1
dilution_ratio = 20 # Prepared plate from normalization script is 10x
final_volume = 800 # Final volume within well.

In [5]:
OT_IVGC.od_normalization(
            file_path = od_file_path,
            target_od = od_target,
            final_volume_uL = final_volume,
            blank = 0,
            threshold_uL = 20,
            dilution_factor = dilution_ratio,
            out_csv = "examples/od.csv"
            )

,well,OD,volume_ul,add_diluent_uL,theoretical_OD,status
0,A1,1.12,86.0,85.714286,1.00,OK
2,A3,1.18,122.0,122.033898,1.00,OK
12,B1,1.10,73.0,72.727273,1.00,OK
14,B3,1.72,335.0,334.883721,1.00,OK
24,C1,1.10,73.0,72.727273,1.00,OK
26,C3,0.10,0.0,0.000000,0.10,below_target
36,D1,1.30,185.0,184.615385,1.00,OK
38,D3,1.22,144.0,144.262295,1.00,OK
48,E1,0.84,0.0,0.000000,0.84,below_target
50,E3,0.76,0.0,0.000000,0.76,below_target


# Generation of randomized layout

In [6]:
OT_IVGC.generate_randomized_mapping(
    filtered_layout,
    media_list = medias,
    replicates = replicate_count,
    seed = 20262,
    out_ot_csv = "examples/cherrypicks.csv",
    out_data_csv = "examples/lq_layout.csv"
)

,IBT #,from_well,replicate,media,source_col,plate,to_well
0,IBT 115000,A1,1,TY,1,plate_1,B4
1,IBT 115000,A1,1,CRE,1,plate_1,B6
2,IBT 115000,A1,4,TY,1,plate_1,C6
3,IBT 115000,A1,4,CRE,1,plate_1,E9
4,IBT 115000,A1,4,LRE,1,plate_1,E10
...,...,...,...,...,...,...,...
187,Blank,H3,2,TY,3,plate_3,E7
188,Blank,H3,3,CRE,3,plate_3,F5
189,Blank,H3,4,LRE,3,plate_3,F12
190,Blank,H3,4,CRE,3,plate_3,G4


# OT-Stamp layout reference

In [7]:
OT_IVGC.stamp_mapping_reference(
    filtered_layout,
    selected_cols = [1,3], # Selected cols when using stamping script.
    out_data_csv = None
)

,IBT #,from_well,well,replicate,deepwell_col
0,IBT 115000,A1,A1,1,1
1,IBT 115000,A1,A2,2,1
2,IBT 115000,A1,A3,3,1
3,IBT 115000,A1,A4,4,1
4,IBT 115002,B1,B1,1,1
...,...,...,...,...,...
59,DH5 alpha,G3,G9,4,3
60,Blank,H3,H6,1,3
61,Blank,H3,H7,2,3
62,Blank,H3,H8,3,3


# Notes / Acknowledgements
As of current script itteration, This protocol is specifically designed around the logic that 1 source col $\rightarrow$ 1 liquid destination plate. <br> 
Stamping plates can currently fit 2 source columns, and are copies of the source column as it is performed using an 8-Channel pipette mount.

This means that this protocols scripts should be altered if:
1. You wish to have fewer/more media or replicates.
2. If your source plate is not designed around having controls in each column.
3. You want to inoculate whole plates instead of being limited to a 6 column source plate.